In [23]:
from cffi.recompiler import recompile
%load_ext autoreload
%autoreload 2

In [24]:
import os
import re
import shutil
import random
from pathlib import Path
from tqdm import tqdm
import numpy as np
import pandas as pd
from yolo_tools import get_yolo_label_df

In [34]:
data_dir = r'/localnvme/data/billboard/fused_data/data7720_mseg_c5_l2_1002'
image_dir = os.path.join(data_dir, 'images')
label_dir = os.path.join(data_dir, 'labels')

bd_dir = r'/localnvme/data/billboard/bd_data/data687_mseg_c6_0917'

defect_list = ['deformation', 'broken', 'abandonment', 'corrosion']

In [26]:
def get_stem2img_dict(img_dir):
    img_list = [img_name for img_name in os.listdir(img_dir)]
    stem_list = [Path(img).stem for img in img_list]
    stem2img_dict = dict(zip(stem_list, img_list))
    return stem2img_dict
def find_defect(label_dir, image_dir, defect_list, exclude_dir=None):
    defect_file_list = []
    exclude_list = os.listdir(exclude_dir) if exclude_dir else None
    label_file_list = os.listdir(label_dir)
    stem2img_dict = get_stem2img_dict(image_dir)
    for label_name in tqdm(label_file_list):
        input_label_path = os.path.join(label_dir, label_name)
        df = get_yolo_label_df(input_label_path, mdet=True, attributes=defect_list)
        with_defect = (df[defect_list] > 0).any().any()
        if with_defect:
            image_name = stem2img_dict[Path(label_name).stem]
            if exclude_list is not None and image_name in exclude_list:
                continue
            defect_file_list.append(image_name)
    return defect_file_list

In [27]:
defect_file_list = find_defect(label_dir, image_dir, defect_list, exclude_dir=bd_dir)

100%|██████████| 7631/7631 [00:44<00:00, 172.48it/s]


In [28]:
len(defect_file_list)

2293

In [39]:
def random_select_defect(data_dir, defect_file_list, save_dir=None, train_ratio=0.9, random_seed=1010, full_path=True, suffix=''):
    image_dir = os.path.join(data_dir, 'images')
    label_dir = os.path.join(data_dir, 'labels')
    file_list = os.listdir(image_dir)
    if label_dir is not None:
        label_list = os.listdir(label_dir)
        label_list = [Path(label_name).stem for label_name in label_list]
        file_list_check = []
        for img_name in tqdm(file_list, desc='img check', total=len(file_list)):
            name = Path(img_name).stem
            if name in label_list:
                file_list_check.append(img_name)
        file_list = file_list_check
    if save_dir is None:
        save_dir = os.path.dirname(image_dir)

    np.random.seed(random_seed)
    np.random.shuffle(file_list)
    val_num = int(len(defect_file_list)*(1-train_ratio))

    val_list = defect_file_list[:val_num]
    train_list = [file_name for file_name in file_list if file_name not in val_list]

    if full_path:
        train_list = [os.path.join(image_dir, name) for name in train_list]
        val_list = [os.path.join(image_dir, name) for name in val_list]

    df_train = pd.DataFrame({'filename': train_list})
    df_val = pd.DataFrame({'filename': val_list})
    df_all = pd.DataFrame({'filename': train_list+val_list})
    df_train.to_csv(os.path.join(save_dir, f'train{suffix}.txt'), header=None, index=None)
    df_val.to_csv(os.path.join(save_dir, f'val{suffix}.txt'), header=None, index=None)
    df_all.to_csv(os.path.join(save_dir, 'all.txt'), header=None, index=None)
    print('%d save to %s,\n%d save to %s!'%(len(train_list), os.path.join(save_dir, f'train{suffix}.txt'),
                                           len(val_list), os.path.join(save_dir, f'val{suffix}.txt')))

In [40]:
random_select_defect(data_dir, defect_file_list, train_ratio=0.7, random_seed=1010, full_path=True, suffix='_70p')
random_select_defect(data_dir, defect_file_list, train_ratio=0.6, random_seed=1010, full_path=True, suffix='_60p')

img check: 100%|██████████| 7631/7631 [00:00<00:00, 23276.64it/s]


6944 save to /localnvme/data/billboard/fused_data/data7720_mseg_c5_l2_1002/train_70p.txt,
687 save to /localnvme/data/billboard/fused_data/data7720_mseg_c5_l2_1002/val_70p.txt!


img check: 100%|██████████| 7631/7631 [00:00<00:00, 21179.25it/s]


6714 save to /localnvme/data/billboard/fused_data/data7720_mseg_c5_l2_1002/train_60p.txt,
917 save to /localnvme/data/billboard/fused_data/data7720_mseg_c5_l2_1002/val_60p.txt!


In [41]:
def random_select_defect_ref(data_dir, ref_list, defect_file_list, save_dir=None, train_ratio=0.9, random_seed=1010, full_path=True, suffix=''):
    image_dir = os.path.join(data_dir, 'images')
    label_dir = os.path.join(data_dir, 'labels')
    file_list = os.listdir(image_dir)
    if label_dir is not None:
        label_list = os.listdir(label_dir)
        label_list = [Path(label_name).stem for label_name in label_list]
        file_list_check = []
        for img_name in tqdm(file_list, desc='img check', total=len(file_list)):
            name = Path(img_name).stem
            if name in label_list:
                file_list_check.append(img_name)
        file_list = file_list_check
    if save_dir is None:
        save_dir = os.path.dirname(image_dir)
    np.random.seed(random_seed)
    np.random.shuffle(file_list)
    val_num = int(len(defect_file_list)*(1-train_ratio))


    intersection_list = list(set(defect_file_list) & set(ref_list))
    remain_list = [file_name for file_name in defect_file_list if file_name not in intersection_list]
    val_select_num = max(0, val_num - len(intersection_list))
    val_select_list = random.sample(remain_list, min(val_select_num, len(remain_list)))

    val_list = val_select_list + intersection_list
    train_list = [file_name for file_name in file_list if file_name not in val_list]

    if full_path:
        train_list = [os.path.join(image_dir, name) for name in train_list]
        val_list = [os.path.join(image_dir, name) for name in val_list]

    df_train = pd.DataFrame({'filename': train_list})
    df_val = pd.DataFrame({'filename': val_list})
    df_all = pd.DataFrame({'filename': train_list+val_list})
    df_train.to_csv(os.path.join(save_dir, f'train{suffix}.txt'), header=None, index=None)
    df_val.to_csv(os.path.join(save_dir, f'val{suffix}.txt'), header=None, index=None)
    df_all.to_csv(os.path.join(save_dir, 'all.txt'), header=None, index=None)
    print('%d save to %s,\n%d save to %s!'%(len(train_list), os.path.join(save_dir, f'train{suffix}.txt'),
                                           len(val_list), os.path.join(save_dir, f'val{suffix}.txt')))

In [42]:
val_80p_ref_path = r'/localnvme/data/billboard/fused_data/data7720_mseg_c6_1002/val_80p_ref.txt'
df = pd.read_csv(val_80p_ref_path, header=None, index_col=False, names=['file_path'])
file_path_list = df['file_path'].to_list()
file_name_list = [Path(file_path).name for file_path in file_path_list]


In [43]:
random_select_defect_ref(data_dir, file_name_list, defect_file_list, train_ratio=0.7, random_seed=1010, full_path=True, suffix='_70p_ref')
random_select_defect_ref(data_dir, file_name_list, defect_file_list, train_ratio=0.6, random_seed=1010, full_path=True, suffix='_60p_ref')

img check: 100%|██████████| 7631/7631 [00:00<00:00, 25417.58it/s]


6944 save to /localnvme/data/billboard/fused_data/data7720_mseg_c5_l2_1002/train_70p_ref.txt,
687 save to /localnvme/data/billboard/fused_data/data7720_mseg_c5_l2_1002/val_70p_ref.txt!


img check: 100%|██████████| 7631/7631 [00:00<00:00, 23138.51it/s]


6714 save to /localnvme/data/billboard/fused_data/data7720_mseg_c5_l2_1002/train_60p_ref.txt,
917 save to /localnvme/data/billboard/fused_data/data7720_mseg_c5_l2_1002/val_60p_ref.txt!


In [52]:
import os
import re
import shutil
from pathlib import Path
from tqdm import tqdm
from datetime import datetime, timedelta

In [15]:
input_dir = r'/localnvme/data/added_data/test_data/test_data_mseg_c6_1021/images'
search_dir = r'/localnvme/data/billboard/fused_data/data7961_seg_c5_1021/images'
cam_name_list = ['cam_DA4930148', 'cam_DA5148680', 'cam_DA5148683', 'cam_DA5324645', 'cam_DA5324655', 'cam_DA6102933']
input_list = ['input1', 'input2', 'input3', 'input4','input5', 'input6']

pattern_1 = r'^camera\d{1}_DA\d{7}_\d{17}.jpg$'
pattern_2 = r'^DA\d{7}_\d{17}.jpg$'
pattern_3 = r'^cam_DA\d{7}_cam_DA\d{7}_\d{17}.jpg$'

In [8]:
# camera5_DA5324655_20250808115951700.jpg
# DA6102933_20250725142226300.jpg
# cam_DA5324655_cam_DA5324655_20250610143015022.jpg


# input_2_DA5324655_20250709154645500.jpg
# cam_300deg_20200523101113000.jpg
# 1590204676.799964046.jpg
# 1590194716.199909210_right.png

In [9]:
file_list = os.listdir(input_dir)

In [12]:
dates = []
for file_name in file_list:
    timestamp = Path(file_name).stem.split('_')[1]
    date = timestamp[:8]
    dates.append(date)
dates = sorted(list(set(dates)))
print(dates)

['20250808', '20250812', '20250829', '20250917', '20250926', '20250930', '20251008', '20251010']


In [17]:
cams, timestamps = [], []
for file_name in file_list:
    cam,timestamp = Path(file_name).stem.split('_')
    cams.append(cam)
    timestamps.append(timestamp)
print(len(cams), len(timestamps))

80 80


In [48]:
timestamps_dt = []
for cam, timestamp in zip(cams, timestamps):
    ts_year = timestamp[:4]
    ts_month = timestamp[4:6]
    ts_day = timestamp[6:8]
    ts_hour = timestamp[8:10]
    ts_min = timestamp[10:12]
    ts_sec = timestamp[12:14]
    ts_msec = timestamp[14:17]
    ts_datetime_str = f'{ts_year}-{ts_month}-{ts_day} {ts_hour}:{ts_min}:{ts_sec}.{ts_msec}'
    timestamp_dt = datetime.strptime(ts_datetime_str, '%Y-%m-%d %H:%M:%S.%f')
    timestamps_dt.append({
        'timestamp':timestamp,
        'start_time': timestamp_dt - timedelta(seconds=5),
        'end_time': timestamp_dt + timedelta(seconds=5),
        'cam':cam,
        'match':[]
    })
print(len(timestamps_dt))

80


In [49]:
pattern_1 = re.compile(pattern_1)
pattern_2 = re.compile(pattern_2)
pattern_3 = re.compile(pattern_3)

match_count = 0
matched_files = []
image_list = os.listdir(search_dir)
for file_name in tqdm(image_list):
    if pattern_1.match(file_name):
        _, cam, timestamp = Path(file_name).stem.split('_')
    elif pattern_2.match(file_name):
        cam, timestamp = Path(file_name).stem.split('_')
    elif pattern_3.match(file_name):
        _, _, _, cam, timestamp = Path(file_name).stem.split('_')
    else:
        continue

    match_count += 1
    file_time = datetime.strptime(timestamp, '%Y%m%d%H%M%S%f')

    for i, ref_ts in enumerate(timestamps_dt):
        if cam==ref_ts['cam'] and ref_ts['start_time'] < file_time < ref_ts['end_time']:
            matched_files.append(file_name)
            timestamps_dt[i]['match'].append(file_name)
print(match_count, len(matched_files))

100%|██████████| 7872/7872 [00:00<00:00, 64899.13it/s]

5213 43


In [54]:
input_image_dir = r'/localnvme/data/billboard/fused_data/data7961_mseg_c6_1022/images'
input_label_dir = r'/localnvme/data/billboard/fused_data/data7961_mseg_c6_1022/labels'
output_image_dir = r'/localnvme/data/added_data/check1022/images'
output_label_dir = r'/localnvme/data/added_data/check1022/labels'

for i, ref_ts in enumerate(timestamps_dt):
    if len(ref_ts['match']) != 0:
        # print(f"{ref_ts['cam']}_{ref_ts['timestamp']}.jpg")
        # print(ref_ts['match'])
        # print()
        for image_name in ref_ts['match']:
            if image_name != f"{ref_ts['cam']}_{ref_ts['timestamp']}.jpg":
                label_name = Path(image_name).stem + '.txt'
                input_image_path = os.path.join(input_image_dir, image_name)
                output_image_path = os.path.join(output_image_dir, image_name)
                input_label_path = os.path.join(input_label_dir, label_name)
                output_label_path = os.path.join(output_label_dir, label_name)
                shutil.copy(input_image_path, output_image_path)
                shutil.copy(input_label_path, output_label_path)
                print(f'copy {image_name}')

copy camera1_DA4930148_20250930165156699.jpg
copy cam_DA5148683_cam_DA5148683_20250829114147299.jpg
copy cam_DA5148683_cam_DA5148683_20250829114150300.jpg
copy DA5148683_20250812150601500.jpg
copy DA5148683_20250812150603600.jpg
copy DA5148683_20250812150604300.jpg
copy DA5148683_20250812150602900.jpg
copy DA5148683_20250812150600800.jpg
copy DA5324645_20250812150750700.jpg
copy camera4_DA5324645_20250812150745300.jpg
copy DA5324645_20250812150747299.jpg
copy camera4_DA5324645_20250812150751899.jpg
copy DA5324645_20250812150751899.jpg
copy camera4_DA5324645_20250812150748400.jpg
copy DA5324645_20250812150745300.jpg
copy DA5148680_20250812150633799.jpg
copy DA5148680_20250812150635599.jpg
copy camera6_DA6102933_20250930165711399.jpg
copy camera1_DA4930148_20250930165156699.jpg
copy camera3_DA5148683_20250812150941100.jpg
copy camera3_DA5148683_20250812150933799.jpg
copy camera3_DA5148683_20250812150938099.jpg
copy DA5148683_20250812150941100.jpg
copy DA5148683_20250812150932399.jpg
copy

In [1]:
import os
from isds_tool.es_tools.reid_tools import images2feature, get_sim


In [2]:
dataset_dir1 = r'/localnvme/data/billboard/fused_data/data7961_mseg_c6_1022'
dataset_dir2 = r'/localnvme/data/added_data/test_data/test_data_mseg_c6_1021_broken_refine'

In [3]:
# images2feature(
#     os.path.join(dataset_dir1, 'result_analysis', 'images_crop'),
#     os.path.join(dataset_dir1, 'result_analysis', 'images_crop_reid_feature'),
#     r'/localnvme/project/dataset_tools/isds_tool/es_tools/reid_model.onnx'
# )

In [3]:
images2feature(
    os.path.join(dataset_dir2, 'result_analysis', 'images_crop'),
    os.path.join(dataset_dir2, 'result_analysis', 'images_crop_reid_feature'),
    r'/localnvme/project/dataset_tools/isds_tool/es_tools/reid_model.onnx'
)

100%|██████████| 103/103 [00:09<00:00, 10.72it/s]


In [4]:
get_sim(
    os.path.join(dataset_dir1, 'result_analysis', 'images_crop_reid_feature'),
    os.path.join(dataset_dir2, 'result_analysis', 'images_crop_reid_feature'),
)

100%|██████████| 103/103 [00:00<00:00, 7084.04it/s]


(57831, 2048) (103, 2048)
(57831, 103)


In [8]:
import pandas as pd

In [9]:
input_dir = r'/localnvme/data/billboard/fused_data/data7961_mseg_c6_1022'
input_image_dir = os.path.join(input_dir, 'images')
input_label_dir = os.path.join(input_dir, 'labels')
output_dir = r'/localnvme/data/added_data/check1022'
output_image_dir = os.path.join(output_dir, 'images')
output_label_dir = os.path.join(output_dir, 'labels')
ref_csv_path = r'/localnvme/data/added_data/test_data/test_data_mseg_c6_1021_broken_refine/result_analysis/data_copy.csv'

In [12]:
df = pd.read_csv(ref_csv_path, header=0, index_col=None)
stem_list = df['file_name'].to_list()

stem2image_dict = get_stem2img_dict(input_image_dir)
stem2label_dict = get_stem2img_dict(input_label_dir)

image_list = [stem2image_dict[stem] for stem in stem_list]
label_list = [stem2label_dict[stem] for stem in stem_list]

1110
